## Step 0 — Imports & Setup

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from google.colab import drive
drive.mount('/content/drive')




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 1 — Load Data
We load the raw CSV and immediately drop the 5 trailing unnamed columns that are entirely empty.

In [3]:
df_raw = pd.read_csv("/content/Pakistan Largest Ecommerce Dataset.csv")

print(f"Raw shape     : {df_raw.shape}")
print(f"Columns       : {df_raw.columns}")

/tmp/ipykernel_21109/2287065364.py:1: DtypeWarning: Columns (1,2,3,7,8,9,11,12,13,14,17,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv("/content/Pakistan Largest Ecommerce Dataset.csv")


Raw shape     : (1048575, 26)
Columns       : Index(['item_id', 'status', 'created_at', 'sku', 'price', 'qty_ordered',
       'grand_total', 'increment_id', 'category_name_1',
       'sales_commission_code', 'discount_amount', 'payment_method',
       'Working Date', 'BI Status', ' MV ', 'Year', 'Month', 'Customer Since',
       'M-Y', 'FY', 'Customer ID', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23',
       'Unnamed: 24', 'Unnamed: 25'],
      dtype='object')


In [4]:
df=df_raw.dropna(axis=1, how='all')
df.shape

(1048575, 21)

In [5]:

df.dropna(subset=df.columns.difference(['item_id']), how='all',inplace=True)
df.shape

/tmp/ipykernel_21109/2028947656.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(subset=df.columns.difference(['item_id']), how='all',inplace=True)


(584524, 21)

In [6]:
df=df.drop(columns=["item_id","sku" , "increment_id","Working Date","BI Status"," MV ","Year","Month","M-Y","FY","Customer ID"])
df

,status,created_at,price,qty_ordered,grand_total,category_name_1,sales_commission_code,discount_amount,payment_method,Customer Since
0,complete,7/1/2016,1950.0,1.0,1950.0,Women's Fashion,\N,0.0,cod,2016-7
1,canceled,7/1/2016,240.0,1.0,240.0,Beauty & Grooming,\N,0.0,cod,2016-7
2,canceled,7/1/2016,2450.0,1.0,2450.0,Women's Fashion,\N,0.0,cod,2016-7
3,complete,7/1/2016,360.0,1.0,60.0,Beauty & Grooming,R-FSD-52352,300.0,cod,2016-7
4,order_refunded,7/1/2016,555.0,2.0,1110.0,Soghaat,\N,0.0,cod,2016-7
...,...,...,...,...,...,...,...,...,...,...
584519,cod,8/28/2018,699.0,1.0,849.0,Women's Fashion,NaN,0.0,cod,2018-8
584520,processing,8/28/2018,35599.0,1.0,35899.0,Mobiles & Tablets,NaN,0.0,bankalfalah,2018-8
584521,processing,8/28/2018,129999.0,2.0,652178.0,Mobiles & Tablets,NaN,0.0,bankalfalah,2018-7
584522,processing,8/28/2018,87300.0,2.0,652178.0,Mobiles & Tablets,NaN,0.0,bankalfalah,2018-7


In [7]:
df_model = df[df["status"].isin(["complete", "canceled"])].copy()
df_model

,status,created_at,price,qty_ordered,grand_total,category_name_1,sales_commission_code,discount_amount,payment_method,Customer Since
0,complete,7/1/2016,1950.0,1.0,1950.0,Women's Fashion,\N,0.0,cod,2016-7
1,canceled,7/1/2016,240.0,1.0,240.0,Beauty & Grooming,\N,0.0,cod,2016-7
2,canceled,7/1/2016,2450.0,1.0,2450.0,Women's Fashion,\N,0.0,cod,2016-7
3,complete,7/1/2016,360.0,1.0,60.0,Beauty & Grooming,R-FSD-52352,300.0,cod,2016-7
5,canceled,7/1/2016,80.0,1.0,80.0,Soghaat,\N,0.0,cod,2016-7
...,...,...,...,...,...,...,...,...,...,...
584493,canceled,8/28/2018,1698.0,1.0,964.0,Appliances,NaN,0.0,bankalfalah,2018-8
584494,canceled,8/28/2018,129999.0,1.0,130299.0,Mobiles & Tablets,NaN,0.0,bankalfalah,2018-8
584495,canceled,8/28/2018,1048.0,1.0,1298.0,Men's Fashion,NaN,0.0,bankalfalah,2018-8
584498,canceled,8/28/2018,3789.0,1.0,5737.0,Appliances,NaN,0.0,bankalfalah,2018-8


In [8]:
df_model.isna().sum()

,0
status,0
created_at,0
price,0
qty_ordered,0
grand_total,0
category_name_1,109
sales_commission_code,88113
discount_amount,0
payment_method,0
Customer Since,5


In [9]:
df_model["category_name_1"] = df_model["category_name_1"].fillna(df_model["category_name_1"].mode()[0])

In [10]:
df_model["created_at"] = pd.to_datetime(
    df["created_at"],
    format="%m/%d/%Y",
    errors="coerce"
)

df_model["Customer Since"] = pd.to_datetime(
    df["Customer Since"],
    format="%Y-%m",
    errors="coerce"
)

df_model["Customer Since"] = df_model["Customer Since"].fillna(df_model["Customer Since"].median())


In [11]:
df_model.isna().sum()

,0
status,0
created_at,0
price,0
qty_ordered,0
grand_total,0
category_name_1,0
sales_commission_code,88113
discount_amount,0
payment_method,0
Customer Since,0


## Step 3 — Define Target Variable: `is_canceled`

### Why `is_canceled`?
Among all status values, only **`complete`** and **`canceled`** are unambiguous terminal states:
- `received`, `order_refunded`, `paid` etc. are intermediate states that may still flip — using them as labels would introduce noise.
- Predicting cancellation is directly actionable: the business can intervene (proactive support, discount offers) **before** revenue is lost.




In [12]:

df_model["is_canceled"] = (df_model["status"] == "canceled").astype(int)
df_model.drop(columns="status", inplace=True)
df_model


,created_at,price,qty_ordered,grand_total,category_name_1,sales_commission_code,discount_amount,payment_method,Customer Since,is_canceled
0,2016-07-01,1950.0,1.0,1950.0,Women's Fashion,\N,0.0,cod,2016-07-01,0
1,2016-07-01,240.0,1.0,240.0,Beauty & Grooming,\N,0.0,cod,2016-07-01,1
2,2016-07-01,2450.0,1.0,2450.0,Women's Fashion,\N,0.0,cod,2016-07-01,1
3,2016-07-01,360.0,1.0,60.0,Beauty & Grooming,R-FSD-52352,300.0,cod,2016-07-01,0
5,2016-07-01,80.0,1.0,80.0,Soghaat,\N,0.0,cod,2016-07-01,1
...,...,...,...,...,...,...,...,...,...,...
584493,2018-08-28,1698.0,1.0,964.0,Appliances,NaN,0.0,bankalfalah,2018-08-01,1
584494,2018-08-28,129999.0,1.0,130299.0,Mobiles & Tablets,NaN,0.0,bankalfalah,2018-08-01,1
584495,2018-08-28,1048.0,1.0,1298.0,Men's Fashion,NaN,0.0,bankalfalah,2018-08-01,1
584498,2018-08-28,3789.0,1.0,5737.0,Appliances,NaN,0.0,bankalfalah,2018-08-01,1


In [13]:
df_model["has_discount"] = (df_model["discount_amount"] > 0).astype(int)
df_model["price_per_unit"] = df_model["grand_total"] / df_model["qty_ordered"].replace(0, np.nan)
df_model["is_cod"] = (df_model["payment_method"].str.lower() == "cod").astype(int)

df_model["has_commission"] = (~df_model["sales_commission_code"].isin([r"\N", None, np.nan])).astype(int)
df_model.drop(columns="sales_commission_code", inplace=True)

df_model["category_name_1"] = df_model["category_name_1"].fillna("Unknown")
le_cat = LabelEncoder()
df_model["category_enc"] = le_cat.fit_transform(df_model["category_name_1"])

df_model["payment_method"] = df_model["payment_method"].fillna("Unknown")
le_pay = LabelEncoder()
df_model["payment_enc"] = le_pay.fit_transform(df_model["payment_method"])


df_model["order_dow"] = df_model["created_at"].dt.dayofweek      # Mon-Sun
df_model["order_month"] = df_model["created_at"].dt.month
df_model["order_year"] = df_model["created_at"].dt.year
df_model["customer_tenure_days"] = (df_model["created_at"] - df_model["Customer Since"]).dt.days   # how long they've been a customer at order time

df_model

,created_at,price,qty_ordered,grand_total,category_name_1,discount_amount,payment_method,Customer Since,is_canceled,has_discount,price_per_unit,is_cod,has_commission,category_enc,payment_enc,order_dow,order_month,order_year,customer_tenure_days
0,2016-07-01,1950.0,1.0,1950.0,Women's Fashion,0.0,cod,2016-07-01,0,0,1950.0,1,0,14,6,4,7,2016,0
1,2016-07-01,240.0,1.0,240.0,Beauty & Grooming,0.0,cod,2016-07-01,1,0,240.0,1,0,1,6,4,7,2016,0
2,2016-07-01,2450.0,1.0,2450.0,Women's Fashion,0.0,cod,2016-07-01,1,0,2450.0,1,0,14,6,4,7,2016,0
3,2016-07-01,360.0,1.0,60.0,Beauty & Grooming,300.0,cod,2016-07-01,0,1,60.0,1,1,1,6,4,7,2016,0
5,2016-07-01,80.0,1.0,80.0,Soghaat,0.0,cod,2016-07-01,1,0,80.0,1,0,12,6,4,7,2016,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
584493,2018-08-28,1698.0,1.0,964.0,Appliances,0.0,bankalfalah,2018-08-01,1,0,964.0,0,0,0,4,1,8,2018,27
584494,2018-08-28,129999.0,1.0,130299.0,Mobiles & Tablets,0.0,bankalfalah,2018-08-01,1,0,130299.0,0,0,9,4,1,8,2018,27
584495,2018-08-28,1048.0,1.0,1298.0,Men's Fashion,0.0,bankalfalah,2018-08-01,1,0,1298.0,0,0,8,4,1,8,2018,27
584498,2018-08-28,3789.0,1.0,5737.0,Appliances,0.0,bankalfalah,2018-08-01,1,0,5737.0,0,0,0,4,1,8,2018,27


In [14]:
df_model

,created_at,price,qty_ordered,grand_total,category_name_1,discount_amount,payment_method,Customer Since,is_canceled,has_discount,price_per_unit,is_cod,has_commission,category_enc,payment_enc,order_dow,order_month,order_year,customer_tenure_days
0,2016-07-01,1950.0,1.0,1950.0,Women's Fashion,0.0,cod,2016-07-01,0,0,1950.0,1,0,14,6,4,7,2016,0
1,2016-07-01,240.0,1.0,240.0,Beauty & Grooming,0.0,cod,2016-07-01,1,0,240.0,1,0,1,6,4,7,2016,0
2,2016-07-01,2450.0,1.0,2450.0,Women's Fashion,0.0,cod,2016-07-01,1,0,2450.0,1,0,14,6,4,7,2016,0
3,2016-07-01,360.0,1.0,60.0,Beauty & Grooming,300.0,cod,2016-07-01,0,1,60.0,1,1,1,6,4,7,2016,0
5,2016-07-01,80.0,1.0,80.0,Soghaat,0.0,cod,2016-07-01,1,0,80.0,1,0,12,6,4,7,2016,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
584493,2018-08-28,1698.0,1.0,964.0,Appliances,0.0,bankalfalah,2018-08-01,1,0,964.0,0,0,0,4,1,8,2018,27
584494,2018-08-28,129999.0,1.0,130299.0,Mobiles & Tablets,0.0,bankalfalah,2018-08-01,1,0,130299.0,0,0,9,4,1,8,2018,27
584495,2018-08-28,1048.0,1.0,1298.0,Men's Fashion,0.0,bankalfalah,2018-08-01,1,0,1298.0,0,0,8,4,1,8,2018,27
584498,2018-08-28,3789.0,1.0,5737.0,Appliances,0.0,bankalfalah,2018-08-01,1,0,5737.0,0,0,0,4,1,8,2018,27


In [15]:
FEATURES = [
    "price", "qty_ordered", "grand_total", "discount_amount",
    "has_discount", "price_per_unit", "is_cod", "category_enc",
    "payment_enc", "has_commission","order_dow","order_month","order_year","customer_tenure_days",
]

df_ml = df_model[FEATURES + ["is_canceled"]].dropna()

X = df_ml[FEATURES]
y = df_ml["is_canceled"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

In [16]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [17]:
x_score = model.score(X_test, y_test)
print(f"  Test Accuracy: {x_score:.4f} ({x_score*100:.2f}%)")

  Test Accuracy: 0.8587 (85.87%)
